# Assignment Template

Please follow the template below to submit part A of your assignment. Make sure all the outputs are provided within the submission file. Make sure to familarise yourself with the assignment brief before.

# Libraries

In [30]:
!pip install transformers accelerate huggingface-hub
!pip install evaluate rouge-score nltk

# All Pre-requsites (Hugging Face login, data import etc.)

Note: If you use a HuggingFace model, please do not share your access key, make sure all the outputs are visible, you will be marked down if outputs are not provided)

Provide all functions that you might need for processing the data and using the model (refer to Unit 4 tutorial for some hints)

In [31]:
from datasets import load_dataset

data = load_dataset("EdinburghNLP/xsum")
data

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [32]:
import evaluate
rouge = evaluate.load("rouge")

In [33]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Google T5 Model
model_name = "google/flan-t5-small"
device = torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Set model to evaluation mode for CPU optimization
model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [34]:
# Highly optimized batch inference function for CPU
def generate_batch_summaries(prompts, batch_size=8):
    predictions = []

    # Process prompts in chunks to manage CPU memory
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]

        # Tokenize the batch. Max length is capped to prevent CPU memory overflow
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=1024,
            return_tensors="pt"
        ).to(device)

        # CPU optimization (saves memory/compute)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False
            )

        # Decode and append to predictions
        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)

    return predictions

In [35]:
samples = data["test"].shuffle(seed=42).select(range(100))
references = [sample["summary"] for sample in samples]

few_shot_examples = ""
for i in range(10):
    example = data['train'][i]
    few_shot_examples += f"Document:\n{example['document']}\nSummary:\n{example['summary']}\n\n"

# Zero-shot Strategy

In [37]:
# Prepare 100 batch prompts for Zero-Shot
prompts_zero = [
    f"Summarize the following text in 1 sentence:\n\n{sample['document']}\n\nSummary:"
    for sample in samples
]

# Generate predictions
predictions_zero = generate_batch_summaries(prompts_zero, batch_size=8)

# Compute ROUGE
results_zero = rouge.compute(predictions=predictions_zero, references=references)

print("Zero-shot Results:", results_zero)
print("Predictions:", len(predictions_zero))
print("References:", len(references))

Zero-shot Results: {'rouge1': np.float64(0.2816904827824066), 'rouge2': np.float64(0.07826928956096015), 'rougeL': np.float64(0.2231149052146858), 'rougeLsum': np.float64(0.22280080278560643)}
Predictions: 100
References: 100


# Few-shot Strategy

In [38]:
# Prepare 100 batch prompts for 10-Shot
prompts_few = [
    f"These are examples from the dataset.\n\n{few_shot_examples}"
    f"Summarize this text according to those examples.\n"
    f"Document:\n{sample['document']}\nSummary:\n"
    for sample in samples
]

# Generate predictions
predictions_few = generate_batch_summaries(prompts_few, batch_size=8)

# Compute ROUGE
results_few = rouge.compute(predictions=predictions_few, references=references)

print("Few-shot Results:", results_few)
print("Predictions:", len(predictions_few))
print("References:", len(references))

Few-shot Results: {'rouge1': np.float64(0.12855009938870746), 'rouge2': np.float64(0.010868922347285703), 'rougeL': np.float64(0.10865597163113458), 'rougeLsum': np.float64(0.10888551026966826)}
Predictions: 100
References: 100


# Prompt Engineering

## Zero-shot

In [39]:
# Prepare 100 batch prompts for Engineered Zero-Shot
prompts_eng_zero = [
    "You are a professional summarization model. You are tasked with summarizing the paragraph so that the wording "
    "stays the same and there are not much changes in the meaning but still cover the important details.\n"
    f"Summarize the following text in 1 sentence:\n\n{sample['document']}\n\nSummary:"
    for sample in samples
]

# Generate predictions
predictions_eng_zero = generate_batch_summaries(prompts_eng_zero, batch_size=8)

# Compute ROUGE
results_eng_zero = rouge.compute(predictions=predictions_eng_zero, references=references)

print("Engineered Zero-shot Results:", results_eng_zero)
print("Predictions:", len(predictions_eng_zero))
print("References:", len(references))

Engineered Zero-shot Results: {'rouge1': np.float64(0.2880868059470616), 'rouge2': np.float64(0.08202618092503017), 'rougeL': np.float64(0.2284142283158016), 'rougeLsum': np.float64(0.22794165019996268)}
Predictions: 100
References: 100


## Few-shot

In [40]:
# Prepare 100 batch prompts for Engineered 10-Shot
prompts_eng_few = [
    "You are a professional summarization model. You are tasked with summarizing the paragraph so that the wording "
    "stays the same and there are not much changes in the meaning but still cover the important details.\n"
    f"Here are the examples:\n\n{few_shot_examples}"
    f"Summarize this text:\nDocument:\n{sample['document']}\nSummary:\n"
    for sample in samples
]

# Generate predictions
predictions_eng_few = generate_batch_summaries(prompts_eng_few, batch_size=8)

# Compute ROUGE
results_eng_few = rouge.compute(predictions=predictions_eng_few, references=references)

print("Engineered Few-shot Results:", results_eng_few)
print("Predictions:", len(predictions_eng_few))
print("References:", len(references))

Engineered Few-shot Results: {'rouge1': np.float64(0.11930819441284757), 'rouge2': np.float64(0.015114101618459717), 'rougeL': np.float64(0.09542264229988809), 'rougeLsum': np.float64(0.09512564244865175)}
Predictions: 100
References: 100


# Evaluation

In [43]:
import pandas as pd

# Compile all results into a DataFrame
results_table = pd.DataFrame({
    "Method": ["Zero-shot", "few-shot", "Engineered Zero-shot", "Engineered few-shot"],
    "ROUGE-1": [
        results_zero["rouge1"],
        results_few["rouge1"],
        results_eng_zero["rouge1"],
        results_eng_few["rouge1"]
    ],
    "ROUGE-2": [
        results_zero["rouge2"],
        results_few["rouge2"],
        results_eng_zero["rouge2"],
        results_eng_few["rouge2"]
    ],
    "ROUGE-L": [
        results_zero["rougeL"],
        results_few["rougeL"],
        results_eng_zero["rougeL"],
        results_eng_few["rougeL"]
    ]
})

# Round to 4 decimal places for readability
results_table = results_table.round(4)

# Display the table
display(results_table)

,Method,ROUGE-1,ROUGE-2,ROUGE-L
0,Zero-shot,0.2817,0.0783,0.2231
1,few-shot,0.1286,0.0109,0.1087
2,Engineered Zero-shot,0.2881,0.0820,0.2284
3,Engineered few-shot,0.1193,0.0151,0.0954


# Results Interpretation (max 200 words)

Note: If you reference something from the model outputs, make sure it can be traced (e.g., if you noticed a interesting pattern in the results, make sure to provide an example of that in the outputs).

The ROUGE score for flan t5 small model are lower than the expected value compared to golden standard, especially in the few shot scenario.

The suspected reason for few shot scores being very low (~0.12) is due to the standard number of examples taking up more space in tokenization and the actual artical that needs to be translated is cut of by truncation parameter. However, zero shot learning achieve significantly high score compare few shot. The average score of zero shot learning is ~0.28 (28%). This is very impressive for a non-fine-tuned model with raw prompt.

Comparing to the benchmark (golden standard) of pre-trained models from famous websites, the zero shot score was low. The zero shot golden standard of fine-tuned flan t5 can successfully reach up to ROUGE score of ~45+ (0.45), while our raw model of flan t5 reached to around ~28(0.28) in zero shot learning.

Benchmark Reference: https://www.google.com/search?q=https://paperswithcode.com/sota/abstractive-text-summarization-on-xsum

Model Card Details: https://huggingface.co/google/flan-t5-small